# 0. Libraries used

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 1. Load and Clean Dataset

In [3]:
print("Loading creditcard.csv...")
df = pd.read_csv("creditcard.csv")
print(f"Original shape: {df.shape}")

# Drop duplicate rows
df_cleaned = df.drop_duplicates()
print(f"Shape after removing duplicates: {df_cleaned.shape}")

# Separate features and target
X = df_cleaned.drop(columns=["Class"])
y = df_cleaned["Class"]

Loading creditcard.csv...
Original shape: (284807, 31)
Shape after removing duplicates: (283726, 31)


# 2. Train-Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} (Fraud: {sum(y_train)})")
print(f"Test: {X_test.shape} (Fraud: {sum(y_test)})")

Train: (226980, 30) (Fraud: 378)
Test: (56746, 30) (Fraud: 95)


# 3. Feature Engineering (NO LEAKAGE)

In [5]:
# Log-transform Amount
X_train["Amount"] = np.log1p(X_train["Amount"])
X_test["Amount"] = np.log1p(X_test["Amount"])

# Scale Amount and Time
scaler_amount = StandardScaler()
X_train["Scaled_Amount"] = scaler_amount.fit_transform(X_train[["Amount"]])
X_test["Scaled_Amount"] = scaler_amount.transform(X_test[["Amount"]])

scaler_time = StandardScaler()
X_train["Scaled_Time"] = scaler_time.fit_transform(X_train[["Time"]])
X_test["Scaled_Time"] = scaler_time.transform(X_test[["Time"]])

# Drop original columns
X_train_model = X_train.drop(columns=["Amount", "Time"])
X_test_model = X_test.drop(columns=["Amount", "Time"])

print("Preprocess completed")

Preprocess completed


# 4. Evaluation Function

In [6]:
def evaluate_model(y_true, y_scores, model_name, use_percentile=False):
    """Evaluate model with optimal threshold tuning."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)
    auprc = average_precision_score(y_true, y_scores)

    best_f1, best_thresh, best_y_pred = 0, 0, None

    # Threshold grid
    if use_percentile:
        thresh_grid = np.percentile(y_scores, np.linspace(90, 99.99, 500))
    else:
        thresh_grid = np.concatenate([
            np.linspace(0.01, 0.99, 200),
            np.percentile(y_scores, np.linspace(0.01, 99.99, 1000))
        ])

    for t in thresh_grid:
        y_pred = (y_scores >= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1, best_thresh, best_y_pred = f1, t, y_pred

    if best_y_pred is None:
        best_y_pred = (y_scores >= np.median(y_scores)).astype(int)
        best_thresh = np.median(y_scores)
        best_f1 = f1_score(y_true, best_y_pred)

    print(f"\n{'='*20} {model_name} {'='*20}")
    print(f"AUPRC: {auprc:.4f}")
    print(f"Best Threshold: {best_thresh:.4f} (F1: {best_f1:.4f})")
    print("\nClassification Report:")
    print(classification_report(y_true, best_y_pred))

    cm = confusion_matrix(y_true, best_y_pred)
    print(f"Confusion Matrix: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")
    print("="*50)

    p = precision_score(y_true, best_y_pred)
    r = recall_score(y_true, best_y_pred)

    return precisions, recalls, auprc, best_f1, p, r, cm

# Phase 1: Supervised Baselines

In [7]:
print("\n" + "="*60)
print("PHASE 1: SUPERVISED BASELINES")
print("="*60)

tree_constraints = {
    "max_depth": 10, "min_samples_split": 20,
    "min_samples_leaf": 10, "class_weight": "balanced", "random_state": 42
}

models = {
    "Decision Tree": DecisionTreeClassifier(**tree_constraints),
    "Linear SVM": LinearSVC(dual=False, class_weight="balanced", random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, **tree_constraints)
}

phase1_results = {}
colors_phase1 = {"Decision Tree": "orange", "Linear SVM": "purple", "Random Forest": "red"}

for name, clf in models.items():
    print(f"\n--- Training {name} ---")
    t0 = time.time()
    clf.fit(X_train_model, y_train)
    print(f"Trained in {time.time() - t0:.2f}s")

    if hasattr(clf, "predict_proba"):
        y_scores = clf.predict_proba(X_test_model)[:, 1]
    else:
        y_scores = clf.decision_function(X_test_model)

    prec, rec, auprc, f1, p, r, cm = evaluate_model(y_test, y_scores, name)
    phase1_results[name] = {
        "precisions": prec, "recalls": rec, "auprc": auprc,
        "f1": f1, "precision": p, "recall": r, "confusion_matrix": cm,
        "color": colors_phase1[name]
    }


PHASE 1: SUPERVISED BASELINES

--- Training Decision Tree ---
Trained in 25.44s

==================== Decision Tree ====================
AUPRC: 0.5812
Best Threshold: 0.9995 (F1: 0.7391)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.76      0.72      0.74        95

    accuracy                           1.00     56746
   macro avg       0.88      0.86      0.87     56746
weighted avg       1.00      1.00      1.00     56746

Confusion Matrix: TN=56630, FP=21, FN=27, TP=68

--- Training Linear SVM ---
Trained in 2.11s

==================== Linear SVM ====================
AUPRC: 0.6675
Best Threshold: 1.0595 (F1: 0.6977)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.62      0.79      0.70        95

    accuracy                           1.00     56746
   macro avg       0.

# Phase 2: Supervised Baselines

In [8]:
print("\n" + "="*60)
print("PHASE 2: ANOMALY DETECTION BASELINES")
print("="*60)

phase2_results = {}

# Isolation Forest
print("\n--- Training Isolation Forest ---")
t0 = time.time()
iforest = IsolationForest(contamination='auto', random_state=42, n_jobs=-1)
iforest.fit(X_train_model)
print(f"Trained in {time.time() - t0:.2f}s")

y_scores_if = -iforest.decision_function(X_test_model)
prec, rec, auprc, f1, p, r, cm = evaluate_model(y_test, y_scores_if, "Isolation Forest", use_percentile=True)
phase2_results["Isolation Forest"] = {
    "precisions": prec, "recalls": rec, "auprc": auprc,
    "f1": f1, "precision": p, "recall": r, "confusion_matrix": cm,
    "color": "blue"
}

# Local Outlier Factor
print("\n--- Training Local Outlier Factor ---")
X_train_normal = X_train_model[y_train == 0]
n_subsample = min(20000, len(X_train_normal))
X_train_normal_sub = X_train_normal.sample(n=n_subsample, random_state=42)
print(f"Using {n_subsample} normal samples for LOF")

t0 = time.time()
lof = LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1)
lof.fit(X_train_normal_sub)
print(f"Trained in {time.time() - t0:.2f}s")

y_scores_lof = -lof.decision_function(X_test_model)
prec, rec, auprc, f1, p, r, cm = evaluate_model(y_test, y_scores_lof, "Local Outlier Factor", use_percentile=True)
phase2_results["Local Outlier Factor"] = {
    "precisions": prec, "recalls": rec, "auprc": auprc,
    "f1": f1, "precision": p, "recall": r, "confusion_matrix": cm,
    "color": "green"
}


PHASE 2: ANOMALY DETECTION BASELINES

--- Training Isolation Forest ---
Trained in 0.57s

==================== Isolation Forest ====================
AUPRC: 0.2125
Best Threshold: 0.1458 (F1: 0.3350)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.31      0.36      0.33        95

    accuracy                           1.00     56746
   macro avg       0.66      0.68      0.67     56746
weighted avg       1.00      1.00      1.00     56746

Confusion Matrix: TN=56577, FP=74, FN=61, TP=34

--- Training Local Outlier Factor ---
Using 20000 normal samples for LOF
Trained in 2.98s


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(



==================== Local Outlier Factor ====================
AUPRC: 0.5484
Best Threshold: 1.4644 (F1: 0.6740)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.71      0.64      0.67        95

    accuracy                           1.00     56746
   macro avg       0.85      0.82      0.84     56746
weighted avg       1.00      1.00      1.00     56746

Confusion Matrix: TN=56626, FP=25, FN=34, TP=61


# Phase 3: SOTA Models (XGBoost + SMOTE + Stacking)

In [9]:
print("\n" + "="*60)
print("PHASE 3: SOTA MODELS")
print("="*60)

phase3_results = {}

# Model A: XGBoost (Imbalanced)
print("\n--- Model A: XGBoost (Imbalanced) ---")
ratio = sum(y_train == 0) / sum(y_train == 1)
print(f"scale_pos_weight: {ratio:.2f}")

t0 = time.time()
xgb_imb = XGBClassifier(scale_pos_weight=ratio, random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_imb.fit(X_train_model, y_train)
print(f"Trained in {time.time() - t0:.2f}s")

y_scores_a = xgb_imb.predict_proba(X_test_model)[:, 1]
prec, rec, auprc, f1, p, r, cm = evaluate_model(y_test, y_scores_a, "A: XGBoost (Imbalanced)")
phase3_results["A: XGBoost (Imbalanced)"] = {
    "precisions": prec, "recalls": rec, "auprc": auprc,
    "f1": f1, "precision": p, "recall": r, "confusion_matrix": cm,
    "color": "black"
}

# Model B: SMOTE + XGBoost
print("\n--- Model B: SMOTE + XGBoost ---")
t0 = time.time()
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_model, y_train)
print(f"Balanced training set: {X_train_res.shape} (Fraud: {sum(y_train_res)})")

xgb_smote = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_smote.fit(X_train_res, y_train_res)
print(f"Trained in {time.time() - t0:.2f}s")

y_scores_b = xgb_smote.predict_proba(X_test_model)[:, 1]
prec, rec, auprc, f1, p, r, cm = evaluate_model(y_test, y_scores_b, "B: SMOTE + XGBoost")
phase3_results["B: SMOTE + XGBoost"] = {
    "precisions": prec, "recalls": rec, "auprc": auprc,
    "f1": f1, "precision": p, "recall": r, "confusion_matrix": cm,
    "color": "crimson"
}

# Model C: SMOTE + XGBoost + iForest Stacking
print("\n--- Model C: SMOTE + XGBoost + iForest Stacking ---")
t0 = time.time()

# Add anomaly score as feature
X_train_stack = X_train_model.copy()
X_train_stack["Anomaly_Score"] = -iforest.decision_function(X_train_model)
X_test_stack = X_test_model.copy()
X_test_stack["Anomaly_Score"] = -iforest.decision_function(X_test_model)

# SMOTE + Train
X_train_stack_res, y_train_stack_res = smote.fit_resample(X_train_stack, y_train)
xgb_stack = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_stack.fit(X_train_stack_res, y_train_stack_res)
print(f"Trained in {time.time() - t0:.2f}s")

y_scores_c = xgb_stack.predict_proba(X_test_stack)[:, 1]
prec, rec, auprc, f1, p, r, cm = evaluate_model(y_test, y_scores_c, "C: SMOTE + XGBoost + iForest")
phase3_results["C: SMOTE + XGBoost + iForest"] = {
    "precisions": prec, "recalls": rec, "auprc": auprc,
    "f1": f1, "precision": p, "recall": r, "confusion_matrix": cm,
    "color": "magenta"
}


PHASE 3: SOTA MODELS

--- Model A: XGBoost (Imbalanced) ---
scale_pos_weight: 599.48
Trained in 7.08s

==================== A: XGBoost (Imbalanced) ====================
AUPRC: 0.8237
Best Threshold: 0.5862 (F1: 0.8639)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.99      0.77      0.86        95

    accuracy                           1.00     56746
   macro avg       0.99      0.88      0.93     56746
weighted avg       1.00      1.00      1.00     56746

Confusion Matrix: TN=56650, FP=1, FN=22, TP=73

--- Model B: SMOTE + XGBoost ---
Balanced training set: (453204, 30) (Fraud: 226602)
Trained in 21.21s

==================== B: SMOTE + XGBoost ====================
AUPRC: 0.8133
Best Threshold: 0.9802 (F1: 0.8521)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.76

# Visualization Functions

In [10]:
def plot_pr_curves(results_dict, title, filename):
    """Plot Precision-Recall curves for a set of results."""
    plt.figure(figsize=(8, 6))
    for name, res in results_dict.items():
        plt.plot(res["recalls"], res["precisions"],
                 label=f"{name} (AUPRC={res['auprc']:.4f})",
                 color=res["color"])
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend(loc="upper right")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {filename}")

def plot_confusion_matrices(results_dict, filename, cols=3):
    """Plot confusion matrices side by side."""
    n = len(results_dict)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
    if rows == 1:
        axes = axes.flatten()
    else:
        axes = axes.ravel()

    for i, (name, res) in enumerate(results_dict.items()):
        disp = ConfusionMatrixDisplay(confusion_matrix=res["confusion_matrix"],
                                      display_labels=["Normal", "Fraud"])
        disp.plot(ax=axes[i], cmap=plt.cm.Blues, values_format='d', colorbar=False)
        axes[i].set_title(name)

    # Hide unused subplots
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {filename}")

def plot_metrics_bar(results_dict, title, filename):
    """Plot metrics comparison bar chart."""
    labels = list(results_dict.keys())
    auprcs = [results_dict[n]["auprc"] for n in labels]
    f1s = [results_dict[n]["f1"] for n in labels]
    precisions = [results_dict[n]["precision"] for n in labels]
    recalls = [results_dict[n]["recall"] for n in labels]

    x = np.arange(len(labels))
    width = 0.18

    plt.figure(figsize=(max(8, len(labels)*3), 6))
    plt.bar(x - 1.5*width, auprcs, width, label='AUPRC', color='navy')
    plt.bar(x - 0.5*width, f1s, width, label='F1-Score', color='crimson')
    plt.bar(x + 0.5*width, precisions, width, label='Precision', color='forestgreen')
    plt.bar(x + 1.5*width, recalls, width, label='Recall', color='darkorange')

    plt.xticks(x, labels, rotation=15, ha='right')
    plt.ylabel('Score')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.ylim(0, 1.05)
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {filename}")

# Phase 1 Visualizations

In [11]:
print("\n--- Phase 1 Visualizations ---")
plot_pr_curves(phase1_results, "Precision-Recall Curve (Phase 1: Supervised Baselines)", "pr_curves_phase1.png")
plot_confusion_matrices(phase1_results, "confusion_matrices_phase1.png")
plot_metrics_bar(phase1_results, "Performance Metrics Comparison (Phase 1)", "metrics_comparison_phase1.png")


--- Phase 1 Visualizations ---
Saved: pr_curves_phase1.png
Saved: confusion_matrices_phase1.png
Saved: metrics_comparison_phase1.png


# Phase 2 Visualizations

In [12]:
print("\n--- Phase 2 Visualizations ---")
plot_pr_curves(phase2_results, "Precision-Recall Curve (Phase 2: Anomaly Detection)", "pr_curves_phase2.png")
plot_confusion_matrices(phase2_results, "confusion_matrices_phase2.png", cols=2)
plot_metrics_bar(phase2_results, "Performance Metrics Comparison (Phase 2)", "metrics_comparison_phase2.png")


--- Phase 2 Visualizations ---
Saved: pr_curves_phase2.png
Saved: confusion_matrices_phase2.png
Saved: metrics_comparison_phase2.png


Phase 3 Visualizations

In [13]:
print("\n--- Phase 3 Visualizations ---")
plot_pr_curves(phase3_results, "Precision-Recall Curve (Phase 3: SOTA Models)", "pr_curves_phase3.png")
plot_confusion_matrices(phase3_results, "confusion_matrices_phase3.png")
plot_metrics_bar(phase3_results, "Performance Metrics Comparison (Phase 3)", "metrics_comparison_phase3.png")


--- Phase 3 Visualizations ---
Saved: pr_curves_phase3.png
Saved: confusion_matrices_phase3.png
Saved: metrics_comparison_phase3.png
